In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# make dummy track_ops object (would be input from command line or gui)
class Track_ops:
    def __init__(self):
        # input list of dataset paths (each contains a 'suite2p' folder)
        self.all_ds_path = [
            'data/jm/jm032/2023-10-19_a/',
            'data/jm/jm032/2023-10-20_a/',
            'data/jm/jm032/2023-10-21_a/'
        ]
        self.save_path = 'data/jm/jm032/tracking/'
        self.reg_chan = 1 # channel to use for registration (0=functional, 1=anatomical) (1 is not always available)

In [ ]:
track_ops = Track_ops()

In [ ]:
# first check how many planes in each dataset
all_n_planes = []

for ds_path in track_ops.all_ds_path:
    # check how many subfolders starting with plane* in suite2p folder
    n_planes = len([name for name in os.listdir(ds_path + 'suite2p') if name.startswith('plane')])
    print(f'Found {n_planes} planes in {ds_path}')
    all_n_planes.append(n_planes)
track_ops.all_n_planes = all_n_planes
# TODO (handling exceptions of no suite2p folder, no plane folders, not matching number of folders etc. )

In [ ]:
def load_all_imgs(track_ops):
    all_ds_avg_ch1 = []
    all_ds_avg_ch2 = []
    all_ds_nchannels = []

    for ds_path in track_ops.all_ds_path:
        ds_nchannels = []
        ds_avg_ch1 = []
        ds_avg_ch2 = []

        for i in range(n_planes):
            ops = np.load(ds_path + 'suite2p/plane' + str(i) + '/ops.npy', allow_pickle=True).item()
            nchannels = ops['nchannels']
            print('nchannels: ' + str(nchannels) + ' for plane ' + str(i) + ' in dataset ' + ds_path)
            ds_avg_ch1.append(ops['meanImg'])
            ds_avg_ch2.append(ops['meanImg_chan2']) if nchannels==2 else ds_avg_ch2.append(None)
            all_ds_nchannels.append(nchannels)

        all_ds_avg_ch1.append(ds_avg_ch1)
        all_ds_avg_ch2.append(ds_avg_ch2)
        all_ds_nchannels.append(ds_nchannels)

    track_ops.all_ds_avg_ch1 = all_ds_avg_ch1
    track_ops.all_ds_avg_ch2 = all_ds_avg_ch2
    track_ops.all_ds_nchannels = all_ds_nchannels

    return all_ds_avg_ch1, all_ds_avg_ch2

In [ ]:
all_ds_avg_ch1, all_ds_avg_ch2 = load_all_imgs(track_ops)

In [ ]:
from skimage.exposure import match_histograms

In [ ]:
# match histograms (using first plane of first dataset as reference)
def match_hist_all(all_ds_avg_ch):
    ref = all_ds_avg_ch[0][0]
    all_ds_avg_ch_matched = []
    for ds_avg_ch in all_ds_avg_ch:
        ds_avg_ch_matched = []
        for i in range(len(ds_avg_ch)):
            ds_avg_ch_matched.append(match_histograms(ds_avg_ch[i], ref))
        all_ds_avg_ch_matched.append(ds_avg_ch_matched)
        
    return all_ds_avg_ch_matched

In [ ]:
track_ops.all_ds_avg_ch1_matched = match_hist_all(all_ds_avg_ch1)
track_ops.all_ds_avg_ch2_matched = match_hist_all(all_ds_avg_ch2)

In [ ]:
# define function to plot all planes of a dataset for a particular channel where rows are planes and columns are datasets
def plot_all_planes(all_ds_avg_ch, n_planes, track_ops, sat_perc=99):
    fig, axs = plt.subplots(n_planes, len(track_ops.all_ds_path), figsize=(10, 10))
    # add dummy dimension to axs if only one plane
    if n_planes==1:
        axs = np.expand_dims(axs, axis=0)
    
    all_ds_avg_ch_matched = match_hist_all(all_ds_avg_ch)


    for i in range(n_planes):
        for j in range(len(track_ops.all_ds_path)):
            img = all_ds_avg_ch_matched[j][i]
            axs[i, j].imshow(img, cmap='gray', vmin=0, vmax=np.percentile(img, sat_perc))
            axs[i, j].set_title('Plane ' + str(i) + ' in dataset ' + str(j))
            axs[i, j].axis('off')
            
    plt.tight_layout()
    plt.show()

In [ ]:
plot_all_planes(all_ds_avg_ch1, n_planes, track_ops)
if track_ops.all_ds_nchannels[0]==2: # if there is a second channel, plot it too
    plot_all_planes(all_ds_avg_ch2, n_planes, track_ops)

In [ ]:
# choose 'reference images' for registration (all except last plane of last dataset) based on which channel is used for registration
if track_ops.reg_chan==0:
    all_ref_img = [all_ds_avg_ch1[i][j] for i in range(len(track_ops.all_ds_path)-1) for j in range(n_planes)]
    all_mov_img = [all_ds_avg_ch1[i][j] for i in range(1, len(track_ops.all_ds_path)) for j in range(n_planes)]
elif track_ops.reg_chan==1:
    all_ref_img = [all_ds_avg_ch2[i][j] for i in range(len(track_ops.all_ds_path)-1) for j in range(n_planes)]
    all_mov_img = [all_ds_avg_ch2[i][j] for i in range(1, len(track_ops.all_ds_path)) for j in range(n_planes)]
    

In [ ]:
# define function that will register two images
from skimage.registration import phase_cross_correlation

def register_img(ref_img, mov_img):
    # register mov_img to ref_img
    shift, _, _ = phase_cross_correlation(ref_img, mov_img)
    # apply shift to mov_img
    mov_img_reg = np.roll(mov_img, shift.astype(int), axis=(0, 1))
    return mov_img_reg



In [ ]:
# apply function to first pair and visualise output
ref_img = all_ref_img[0]
mov_img = all_mov_img[0]
mov_img_reg = register_img(ref_img, mov_img)


In [ ]:
# define a function that will take two images and return an rgb image
def make_rgb_img(img1, img2):
    img_rgb = np.zeros((img1.shape[0], img1.shape[1], 3))
    # normalise images to 0-1
    img1_norm = (img1 - np.min(img1)) / (np.max(img1) - np.min(img1))
    img2_norm = (img2 - np.min(img2)) / (np.max(img2) - np.min(img2))
    img_rgb[:, :, 0] = img1_norm
    img_rgb[:, :, 1] = img2_norm
    
    return img_rgb

In [ ]:
# compute affine transformation matrix to register the two images
from skimage.transform import AffineTransform, warp

def compute_affine_transform(ref_img, mov_img):
    # compute transformation matrix
    transform = AffineTransform()
    transform.estimate(ref_img, mov_img)
    return transform




In [ ]:
# apply affine transformation to mov_img
transform = compute_affine_transform(ref_img, mov_img)
# mov_img_reg = warp(mov_img, transform.inverse)

In [ ]:
# plot rgb image of ref and mov images  
img_rgb = make_rgb_img(ref_img, mov_img)
img_rgb_reg = make_rgb_img(ref_img, mov_img_reg)

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(img_rgb)
axs[0].set_title('Before registration')
axs[1].imshow(img_rgb_reg)
axs[1].set_title('After registration')